In [38]:
import pandas as pd
import json
import re
import anthropic
from groq import Groq
import os
from dotenv import load_dotenv

load_dotenv()
anthropic_client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))

groq_client = Groq(
    api_key=os.environ.get("GROQ_API_KEY"),
)

chat_completion = groq_client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": "Explain the importance of fast language models",
        }
    ],
    model="llama-3.3-70b-versatile",
)

print(chat_completion.choices[0].message.content)

print("all good")

INFO:httpx:HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


Fast language models are crucial in today's technological landscape, and their importance can be seen in several areas:

1. **Improved User Experience**: Fast language models enable applications to respond quickly to user queries, providing a seamless and efficient user experience. This is particularly important for real-time applications, such as conversational AI, chatbots, and virtual assistants.
2. **Increased Productivity**: Fast language models can process large amounts of data quickly, allowing businesses and organizations to automate tasks, analyze data, and make decisions more rapidly. This can lead to increased productivity, reduced costs, and improved competitiveness.
3. **Enhanced Customer Service**: Fast language models can help companies provide 24/7 customer support, answering frequent questions, and routing complex issues to human representatives. This can improve customer satisfaction, reduce wait times, and enhance the overall customer experience.
4. **Competitive Adv

In [26]:

#generate tweet prompt

def generate_tweets(n):
    prompt = f"""Generate {n} synthetic tweets as a JSON array. Each tweet must conform exactly to this schema:

{{
  "id": "<unique numeric string>",
  "text": "<tweet text, max 280 chars>",
  "created_at": "<ISO 8601 timestamp>",
  "entities": {{
    "urls": [{{"expanded_url": "<full url or empty list>"}}],
    "mentions": [{{"username": "<handle without @>"}}],
    "hashtags": [{{"tag": "<hashtag without #>"}}]
  }},
  "author": {{
    "id": "<unique numeric string>",
    "username": "<twitter handle>",
    "name": "<display name>",
    "verified": <true|false>,
    "created_at": "<ISO 8601 timestamp>",
    "public_metrics": {{
      "followers_count": <int>,
      "following_count": <int>,
      "tweet_count": <int>
    }}
  }},
  "public_metrics": {{
    "retweet_count": <int>,
    "reply_count": <int>,
    "like_count": <int>,
    "quote_count": <int>
  }},
  "label": <1 for scam, 0 for legitimate, -1 for ambiguous>
}}

Generate in this distribution:
- 40% scam (label: 1): crypto giveaway scams, impersonation of Elon Musk/Vitalik/CZ,
  "send ETH get 2x back" schemes, phishing links, urgent limited-time offers.
  Fresh accounts: low followers, high following, young account age, suspicious usernames.
- 40% legitimate (label: 0): real crypto discussion, news sharing, price commentary,
  developer posts. Aged profiles, balanced follower ratios, occasionsally verified.
- 20% ambiguous (label: -1): aggressive but legitimate marketing, new accounts posting
  real content, promotional language that isn't technically a scam.

Make metadata internally consistent — a 3-day-old account should have low tweet_count,
scam tweets should have inflated likes but near-zero replies.
Vary writing style, don't repeat phrases across tweets.
Return ONLY the raw JSON array, no explanation or markdown."""

    return prompt


In [ ]:
#tweets file generation

try: 
    with open("data/tweets.json", "r") as f:
        tweets = json.load(f)
    print(f"loaded {len(tweets)} existing tweets")
except FileNotFoundError:
    tweets = []

for _ in range(10):
    message = anthropic_client.messages.create(
        model="claude-opus-4-5",
        max_tokens=8096,
        messages=[{"role": "user", "content": generate_tweets(10)}]
    )
    response_text = message.content[0].text
    response_text = response_text.strip().removeprefix("```json").removesuffix("```").strip()
    tweets += json.loads(response_text)
    print(f"Total so far: {len(tweets)}")

with open("data/tweets.json", "w") as f:
    json.dump(tweets, f, indent=2)
    

print(f"Final Total: {len(tweets)}")

In [5]:
from snorkel.labeling import labeling_function
import re

SCAM = 1
LEGIT = 0
ABSTAIN = -1


In [61]:
from datetime import datetime, timezone
import time
suspicious_domains = re.compile(
    r"(bit\.ly|tinyurl|t\.co|goo\.gl|ow\.ly|short\.io|rebrand\.ly)"
    )
suspicious_keywords = re.compile(
    r"(giveaway|claim|free|airdrop|crypto|reward|promo|win|bonus)"
    )
@labeling_function()
def lf_giveaway_keywords(x):
    keywords = ["giveaway", "giving back", "send", "get back", "2x", "double" "free"]
    text = x["text"].lower()
    if any(kw in text for kw in keywords):
        return SCAM
    return ABSTAIN

@labeling_function()
def lf_urgency_language(x):
    keywords = ["limited time", "running out", "last chance", "hurry", "act now"]
    text = x['text'].lower()
    if any(kw in text for kw in keywords):
        return SCAM
    return ABSTAIN

@labeling_function()
def lf_new_account(x): 
    followers = int(x["author"]["public_metrics"]["followers_count"])
    created = datetime.fromisoformat(x["author"]["created_at"].replace("Z", "+00:00"))
    age_days = (datetime.now(timezone.utc) - created).days
    tweet_count = int(x['author']['public_metrics']['tweet_count'])

    if (age_days < 30 or tweet_count < 50) and followers < 100: 
        return SCAM
    return ABSTAIN

@labeling_function()
def lf_follower_ratio(x):

    followers_count = x["author"]["public_metrics"]["followers_count"]
    following_count = x["author"]["public_metrics"]["following_count"]

    if following_count > followers_count * 10:
        return SCAM
    if followers_count > followers_count * 100:
        return LEGIT
    return ABSTAIN

@labeling_function()
def lf_crypto_keywords(x):
    text = x['text'].lower()
    crypto_terms = [
        # Bitcoin
        "btc", "bitcoin", "satoshi", "sats",
        # Ethereum
        "eth", "ethereum", "wei", "gwei",
        # Other major coins
        "doge", "dogecoin", "solana", "sol", "xrp", "ripple", "cardano", "ada",
        "litecoin", "ltc", "bnb", "binance", "avax", "avalanche", "matic", "polygon",
        "shib", "shiba", "pepe", "floki"]
    scam_terms = [# Scam-specific crypto language
        "wallet", "blockchain", "crypto", "token", "coin", "defi", "nft",
        "airdrop", "whitelist", "presale", "mint", "claim", "reward",
        "hodl", "moon", "lambo", "ape in", "to the moon"
        ]
    if any(kw in text for kw in crypto_terms) and any(kw in text for kw in scam_terms):
        return SCAM
    return ABSTAIN

@labeling_function()
def lf_impersonation_text(x):
    text = x['text'].lower()
    keywords = ["official", "real", "verified", "ceo"]
    if any(kw in text for kw in keywords):
        return SCAM
    return ABSTAIN

@labeling_function()
def lf_suspicious_url(x):
    urls = x['entities']['urls']
    if not urls:
        return ABSTAIN
    url = urls[0]['expanded_url'].lower()

    if suspicious_domains.search(url) or suspicious_keywords.search(url):
        return SCAM
    return ABSTAIN

@labeling_function()
def lf_engagement_anomaly(x):
    like_count = x['public_metrics']['like_count']
    reply_count = x['public_metrics']['reply_count']

    if like_count > 1000 and reply_count < 5:
        return SCAM
    return ABSTAIN

@labeling_function()
def lf_suspicious_handle(x):
    handle = x['author']['username']
    suspicious_handle_keywords = ['official', 'real', 'verified']
    if any(kw in handle for kw in suspicious_handle_keywords):
        return SCAM
    return ABSTAIN

@labeling_function()
def lf_llm_check(x):
    time.sleep(1)
    x = {k: v for k, v in x.items() if k != 'label'}
    prompt = f"considering that the following is a tweet (from the twitter api V2), along with the appropriate metadata, examine the potential that this tweet could be a crypto scam, or some sort of way of defrauding the reader. Consider the material of their text, as well as the patterns presented by the metadata, such as the follower count, usernames, etc. if the following seems to be a scam, return 'SCAM' as your response, return 'LEGIT' if this seems genuine. If unsure, return 'ABSTAIN'. It's very important that you limit your response to one of those answers:f{x}"
    completion = groq_client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        model="llama-3.1-8b-instant",
    )
    response = completion.choices[0].message.content.lower()
    if "scam" in response:
        return SCAM
    elif "legit" in response:
        return LEGIT
    return ABSTAIN

@labeling_function()
def lf_verified_legit(x):
    verified = x['author']['verified']
    if verified:
        return LEGIT
    return ABSTAIN


In [65]:
from snorkel.labeling import PandasLFApplier

lfs = [
    lf_giveaway_keywords,
    lf_urgency_language,
    lf_new_account,
    lf_follower_ratio,
    lf_crypto_keywords,
    lf_impersonation_text,
    lf_suspicious_url,
    lf_engagement_anomaly,
    lf_suspicious_handle,
    # lf_llm_check
    lf_verified_legit
]

with open("data/tweets.json", "r") as f:
        tweets = json.load(f)

df = pd.DataFrame(tweets)

applier = PandasLFApplier(lfs = lfs)

L_train = applier.apply(df)

print(L_train.shape)

100%|██████████| 400/400 [00:00<00:00, 22174.49it/s]

(400, 10)


In [66]:
from snorkel.labeling import LFAnalysis 

LFAnalysis(L = L_train, lfs = lfs).lf_summary()

/Users/yeongbinlee/code/crypto_scam_detector/venv/lib/python3.13/site-packages/snorkel/labeling/analysis.py:61: FutureWarning: Input has data type int64, but the output has been cast to float64.  In the future, the output data type will match the input. To avoid this warning, set the `dtype` parameter to `None` to have the output dtype match the input, or set it to the desired output data type.
  m = sparse.diags(np.ravel(self._L_sparse.max(axis=1).todense()))


,j,Polarity,Coverage,Overlaps,Conflicts
lf_giveaway_keywords,0,[1],0.2350,0.2350,0.0075
lf_urgency_language,1,[1],0.1225,0.1200,0.0000
lf_new_account,2,[1],0.2075,0.1900,0.0000
lf_follower_ratio,3,[1],0.3450,0.3450,0.0000
lf_crypto_keywords,4,[1],0.4175,0.3425,0.0350
lf_impersonation_text,5,[1],0.1825,0.1475,0.0200
lf_suspicious_url,6,[1],0.2950,0.2875,0.0075
lf_engagement_anomaly,7,[1],0.3350,0.3325,0.0000
lf_suspicious_handle,8,[1],0.0375,0.0325,0.0000
lf_verified_legit,9,[0],0.1925,0.0625,0.0625


In [67]:
from snorkel.labeling.model import LabelModel

label_model = LabelModel(cardinality=2, verbose= True)
label_model.fit(L_train=L_train, n_epochs= 500, lr = 0.001, seed =42)

INFO:root:Computing O...
INFO:root:Estimating \mu...
  0%|          | 0/500 [00:00<?, ?epoch/s]INFO:root:[0 epochs]: TRAIN:[loss=1.522]
INFO:root:[10 epochs]: TRAIN:[loss=1.423]
INFO:root:[20 epochs]: TRAIN:[loss=1.221]
INFO:root:[30 epochs]: TRAIN:[loss=0.970]
INFO:root:[40 epochs]: TRAIN:[loss=0.708]
INFO:root:[50 epochs]: TRAIN:[loss=0.467]
INFO:root:[60 epochs]: TRAIN:[loss=0.278]
INFO:root:[70 epochs]: TRAIN:[loss=0.156]
INFO:root:[80 epochs]: TRAIN:[loss=0.092]
INFO:root:[90 epochs]: TRAIN:[loss=0.066]
INFO:root:[100 epochs]: TRAIN:[loss=0.056]
INFO:root:[110 epochs]: TRAIN:[loss=0.053]
INFO:root:[120 epochs]: TRAIN:[loss=0.050]
INFO:root:[130 epochs]: TRAIN:[loss=0.047]
INFO:root:[140 epochs]: TRAIN:[loss=0.045]
INFO:root:[150 epochs]: TRAIN:[loss=0.042]
INFO:root:[160 epochs]: TRAIN:[loss=0.040]
INFO:root:[170 epochs]: TRAIN:[loss=0.038]
INFO:root:[180 epochs]: TRAIN:[loss=0.036]
INFO:root:[190 epochs]: TRAIN:[loss=0.034]
INFO:root:[200 epochs]: TRAIN:[loss=0.032]
INFO:root:[21

In [55]:
probs = label_model.predict_proba(L = L_train)
print(probs)

[[2.00530404e-12 1.00000000e+00]
 [5.00000000e-01 5.00000000e-01]
 [5.39703380e-08 9.99999946e-01]
 [5.00000000e-01 5.00000000e-01]
 [1.12512907e-01 8.87487093e-01]
 [2.54681662e-01 7.45318338e-01]
 [7.86729818e-13 1.00000000e+00]
 [2.54681662e-01 7.45318338e-01]
 [2.70610180e-01 7.29389820e-01]
 [7.86189729e-09 9.99999992e-01]
 [6.85229570e-13 1.00000000e+00]
 [5.00000000e-01 5.00000000e-01]
 [1.40916918e-13 1.00000000e+00]
 [2.54681662e-01 7.45318338e-01]
 [2.54681662e-01 7.45318338e-01]
 [3.37737781e-11 1.00000000e+00]
 [2.70610180e-01 7.29389820e-01]
 [1.22623364e-04 9.99877377e-01]
 [5.00000000e-01 5.00000000e-01]
 [1.75282929e-06 9.99998247e-01]
 [5.25213451e-16 1.00000000e+00]
 [5.00000000e-01 5.00000000e-01]
 [7.21288110e-11 1.00000000e+00]
 [5.00000000e-01 5.00000000e-01]
 [8.78356845e-02 9.12164315e-01]
 [2.23238427e-09 9.99999998e-01]
 [1.12512907e-01 8.87487093e-01]
 [4.47969085e-04 9.99552031e-01]
 [2.54681662e-01 7.45318338e-01]
 [2.68832319e-13 1.00000000e+00]
 [5.702488